# 02. 모델 학습 — 빌트인 XGBoost

## 이 노트북에서 배우는 것
- SageMaker **빌트인 알고리즘 컨테이너**(XGBoost) 이미지를 가져오기
- `Estimator` 로 학습 작업(Training Job) 구성 — 하이퍼파라미터 지정
- `train` / `validation` 채널을 지정하고 `fit()` 실행
- 학습된 **모델 아티팩트(model.tar.gz)** 위치 확인

전처리 노트북(`01`)을 먼저 실행했어야 합니다.

## 0. 환경 복원

In [ ]:
import sagemaker
from sagemaker import get_execution_role

sess = sagemaker.Session()
role = get_execution_role()
region = sess.boto_region_name

# 01 노트북에서 저장한 값 복원
%store -r bucket prefix train_s3 validation_s3 test_x_s3 test_y_s3
print("train:", train_s3)
print("valid:", validation_s3)

## 1. XGBoost 컨테이너 이미지 가져오기

SageMaker는 알고리즘별 관리형 도커 이미지를 제공합니다. 리전에 맞는 이미지 URI를 조회합니다.

In [ ]:
from sagemaker import image_uris

# TODO: XGBoost 빌트인 컨테이너 이미지를 조회하세요.
#  - framework 이름은 무엇일까요? version 은 "1.7-1" 을 사용합니다.
#  - 참고: https://sagemaker.readthedocs.io/en/v2.219.0/api/utility/image_uris.html
xgb_image_uri = image_uris.retrieve(framework=____, region=region, version="1.7-1")
print(xgb_image_uri)

## 2. Estimator 구성

`Estimator` 는 "어떤 이미지를, 어떤 인스턴스에서, 어떤 하이퍼파라미터로 학습할지"를 정의합니다.

> 우리 문제는 3개 클래스 분류입니다. XGBoost에서 다중 클래스 확률을 출력하려면
> `objective="multi:softprob"` 과 `num_class` 를 지정해야 합니다.

In [ ]:
from sagemaker.estimator import Estimator

model_output = f"s3://{bucket}/{prefix}/models"

xgb = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    # TODO: 학습 인스턴스 타입을 지정하세요 (예: "ml.m5.xlarge").
    instance_type=____,
    output_path=model_output,
    sagemaker_session=sess,
    base_job_name="student-xgb",
)

# TODO: 다중 클래스(3개) 확률 출력을 위한 하이퍼파라미터를 채우세요.
#  - objective: "multi:softprob"
#  - num_class: 클래스 개수
#  - num_round: 부스팅 라운드 수 (예: 100)
#  - 참고: https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost_hyperparameters.html
xgb.set_hyperparameters(
    objective=____,
    num_class=____,
    num_round=____,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    eval_metric="mlogloss",
)

## 3. 학습 채널 지정 & 실행

빌트인 XGBoost는 CSV 입력을 받습니다. `train`, `validation` 두 채널을 넘깁니다. (학습에 수 분 소요)

In [ ]:
from sagemaker.inputs import TrainingInput

# TODO: 입력 채널의 content_type 을 지정하세요 (CSV).
train_input = TrainingInput(train_s3, content_type=____)
val_input = TrainingInput(validation_s3, content_type=____)

# TODO: fit 에 넘기는 딕셔너리의 키가 곧 "채널 이름"입니다.
#       학습 채널 이름과 검증 채널 이름을 각각 지정하세요.
xgb.fit({____: train_input, ____: val_input})

## 4. 학습 곡선 시각화

`TrainingJobAnalytics` 는 CloudWatch 로그에서 매 라운드의 지표 값을 추출합니다. Debugger 없이도 학습/검증 오차의 변화를 확인할 수 있습니다.

In [ ]:
import re
import boto3
import matplotlib.pyplot as plt
%matplotlib inline

# CloudWatch Logs에서 매 라운드의 지표를 추출합니다.
job_name = xgb.latest_training_job.name
logs_client = boto3.client("logs", region_name=region)
log_group = "/aws/sagemaker/TrainingJobs"

# 로그 스트림 찾기
streams = logs_client.describe_log_streams(
    logGroupName=log_group,
    logStreamNamePrefix=job_name,
)["logStreams"]

train_loss, val_loss = [], []
pattern = re.compile(r"\[(\d+)\].*?train-mlogloss:([\d.]+).*?validation-mlogloss:([\d.]+)")

for stream in streams:
    token = None
    while True:
        kwargs = {"logGroupName": log_group, "logStreamName": stream["logStreamName"], "startFromHead": True}
        if token:
            kwargs["nextToken"] = token
        resp = logs_client.get_log_events(**kwargs)
        for event in resp["events"]:
            m = pattern.search(event["message"])
            if m:
                rnd = int(m.group(1))
                train_loss.append((rnd, float(m.group(2))))
                val_loss.append((rnd, float(m.group(3))))
        if resp["nextForwardToken"] == token:
            break
        token = resp["nextForwardToken"]

if not train_loss:
    print("WARNING: No per-round metrics found in logs yet. Wait a minute and re-run.")
else:
    train_loss.sort(); val_loss.sort()
    rounds_t, vals_t = zip(*train_loss)
    rounds_v, vals_v = zip(*val_loss)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(rounds_t, vals_t, label="train mlogloss")
    ax.plot(rounds_v, vals_v, label="validation mlogloss")
    ax.set_xlabel("Boosting round")
    ax.set_ylabel("mlogloss")
    ax.set_title("Training Curve - train vs validation loss")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. 학습 결과 저장

학습 작업 이름과 모델 아티팩트(S3) 위치를 다음 노트북용으로 저장합니다.

In [ ]:
training_job_name = xgb.latest_training_job.name
model_data = xgb.model_data
print("training job :", training_job_name)
print("model.tar.gz :", model_data)

%store xgb_image_uri training_job_name model_data

> 로그에서 `validation-mlogloss` 값이 라운드마다 감소하는 것을 확인해 보세요.
> 이 값이 뒤의 **하이퍼파라미터 튜닝(04)** 에서 최소화할 목표 지표가 됩니다.

✅ **완료!** 다음은 `03_evaluation.ipynb` — 테스트셋으로 모델 성능을 평가합니다.